In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH75=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH75_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH75_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH75[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [ ]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 25 < x < 125]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts/(len(x_easy)*bin_width)   # Densità normalizzata
yerr=np.sqrt(bin_counts)/(len(x_easy)*bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=75, sigma=5, gamma=1, norm2=1, mu2=75, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (50, 100)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()

#print(sum(bin_densities))
#print(len(x_easy))

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 578 (χ²/ndof = 13.4)       │              Nfcn = 432              │
│ EDM = 2.91e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.644   │   0.019   │            │            │         │         │       │
│ 1 │ mu     │   74.72   │   0.12    │            │            │   50    │   100   │       │
│ 2 │ sigma  │   8.20    │   0.15    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │   2.87    │   0.12    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.369   │   0.019   │            │            │         │         │       │
│ 5 │ mu2    │   77.96   │   0.09    │            │            │         │         │       │
│ 6 │ sigma2 │   4.85    │   0.11    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────────────┐
│        │     norm       mu    sigma    gamma    norm2      mu2   sigma2   gamma2 │
├────────┼─────────────────────────────────────────────────────────────────────────┤
│   norm │ 0.000364   1.7e-3  -1.6e-3  -1.2e-3 -0.35e-3   0.8e-3  -1.8e-3        0 │
│     mu │   1.7e-3   0.0148   -0.004   -0.008 -1.67e-3   -0.000   -0.009    0.000 │
│  sigma │  -1.6e-3   -0.004   0.0216   -0.005  1.48e-3   -0.008    0.005    0.000 │
│  gamma │  -1.2e-3   -0.008   -0.005   0.0146  1.29e-3   -0.001    0.008    0.000 │
│  norm2 │ -0.35e-3 -1.67e-3  1.48e-3  1.29e-3 0.000354 -0.75e-3  1.83e-3        0 │
│    mu2 │   0.8e-3   -0.000   -0.008   -0.001 -0.75e-3  0.00798   -0.003    0.000 │
│ sigma2 │  -1.8e-3   -0.009    0.005    0.008  1.83e-3   -0.003   0.0127    0.000 │
│ gamma2 │        0    0.000    0.000    0.000        0    0.000    0.000        0 │
└────────┴─────────────────────────────────────────────────────────────────────────┘

In [3]:
fit_MH75_values={}
fit_MH75_errors={}

fit_values={'MH75': fit_MH75_values,}
fit_errors={'MH75_errors': fit_MH75_errors}



for param in m_voigt.parameters:
    fit_MH75_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH75_errors[error] = m_voigt.errors[error]

print(fit_MH75_values)
print(fit_MH75_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH75"]=fit_MH75_values
results["MH75_errors"]=fit_MH75_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH75"]=fit_MH75_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH75_errors"]=fit_MH75_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.6444701712721482, 'mu': 74.71803096180155, 'sigma': 8.198288025790674, 'gamma': 2.866367690311915, 'norm2': 0.3688330898017668, 'mu2': 77.95596907321944, 'sigma2': 4.85294864422987, 'gamma2': 0.001}
{'norm': 0.019080083722452252, 'mu': 0.12179740268040007, 'sigma': 0.14706235678016633, 'gamma': 0.12092083767338035, 'norm2': 0.018806214568614463, 'mu2': 0.08934072117532924, 'sigma2': 0.11259480529875948, 'gamma2': 1e-05}
